In [1062]:
import csv
import random
import math

In [1063]:
def layer_pass(input_layer, weights):
    output_layer_size = len(weights)
    output_layer_sums = [0 for i in range(output_layer_size)]
    for o in range(output_layer_size):
        for i in range(len(input_layer)):
            output_layer_sums[o] += weights[o][i] * input_layer[i]
    return output_layer_sums

In [1064]:
def apply_activation_function(sums, mutator):
    return [mutator(x) for x in sums]
    
def relu(val):
    if val < 0:
        return 0
    return val
def relu_p(val):
    if val > 0:
        return 1
    return 0
    
def sigmoid(val):
    return 1 / (1 + math.exp(-1 * val))
def sigmoid_p(val):
    s = val
    return s * (1 - s)
    
def tanh(val):
    return math.tanh(val)
def tanh_p(val):
    return 1 - math.pow(val, 2)
    
def one_to_one(val):
    return val

def leaky_relu(val):
    if val > 0:
        return val
    return 0.01 * val
def leaky_relu_p(val):
    if val > 0:
        return 1
    return 0.01

In [1065]:
def create_network_weights(layer_sizes, modifier = 0.1):
    weights = [[[((random.random() * 2 - 1) * modifier) for i in range(layer_sizes[l-1])] for c in range(layer_sizes[l])] for l in range(1, len(layer_sizes))]
    return weights

In [1066]:
def pass_values(input_values, network_weights, functions):
    values = input_values
    activation_values = []
    for l in range(len(network_weights)):
        values = layer_pass(values, network_weights[l])
        if l != len(network_weights) - 1:
            values = apply_activation_function(values, functions[l])
        activation_values.append(values)
    return activation_values

In [1067]:
def softmax(values):
    max_value = max(values)
    values = [math.exp(x - max_value) for x in values]
    values_sum = sum(values)
    values = [x / values_sum for x in values]
    return values

In [1068]:
def cross_entropy_loss(prediction, actual):
    return -1 * sum([actual[x] * math.log(prediction[x]) for x in range(len(actual))])

In [1069]:
def error_gradient(prediction, actual):
    return [prediction[x] - actual[x] for x in range(len(actual))]

In [1070]:
def back_propogate_layer(error, activation_values, weights, learning_rate, activation_function_derivative):
    gradient_matrix = output_gradients(error, activation_values)
    updated_weights = update_weights(weights, learning_rate, gradient_matrix)

    # Multiply internal layer error buy the activation values of the prior layer after being passed through the derivative
    internal_layer_error = layer_error(weights, error)
    derived_function_activation_values = apply_activation_function(activation_values, activation_function_derivative)
    new_error = [internal_layer_error[i]  * derived_function_activation_values[i] for i in range(len(internal_layer_error))]
    
    return new_error, updated_weights

In [1071]:
def back_propogate_network(input_values, network_weights, functions, functions_p, output_values):
    activation_values = pass_values(input_values, network_weights, functions)
    network_output = activation_values[-1]
    softmaxed_values = softmax(network_output)
    loss = cross_entropy_loss(softmaxed_values, output_values)
    error = error_gradient(softmaxed_values, output_values)
    error_i = error

    new_weights = [0 for l in range(len(network_weights))]

    for i in range(-1, -1-len(network_weights), -1):
        layer_weights = network_weights[i]
        
        activation_sums = []
        if i != -len(network_weights):
            activation_sums = activation_values[i-1]
        else:
             activation_sums = input_values

        layer_function = one_to_one
        if i != -len(network_weights):
            layer_function = functions_p[i]
        error, new_layer_weights = back_propogate_layer(error, activation_sums, layer_weights, 0.01, layer_function)
        new_weights[i] = new_layer_weights

    return new_weights, loss, error_i
    

In [1072]:
def output_gradients(error, activation_values):
    return[[e * v for v in activation_values] for e in error]

In [1073]:
def update_weights(weights, learning_rate, gradient_matrix):
    return [[weights[n][c] - (gradient_matrix[n][c] * learning_rate) for c in range(len(weights[n]))] for n in range(len(weights))]

In [1074]:
def layer_error(weights, error):
    return [sum([error[i] * weights[i][e] for i in range(len(weights))]) for e in range(len(weights[0]))]

In [1075]:
def test_network(test_x, test_y, network_weights):
    evaluations = [0,0]
    for i in range(len(test_x)):
        prediction = softmax(pass_values(test_x[i], network_weights, functions)[-1])
        if test_y[i][prediction.index(max(prediction))] == 1:
            evaluations[0] += 1
        else:
            evaluations[1] += 1
    return evaluations

# Begin network

## Prepare Data

In [986]:
type_map = {
    "setosa": 0,
    "versicolor": 1,
    "virginica": 2
}
data = []

with open('iris.csv', newline='') as csvfile:
    spamreader = csv.reader(csvfile, delimiter=' ', quotechar='|')
    for row in spamreader:
        split_data = (row[0].split(","))
        try:
            output_row = [float(split_data[0]),float(split_data[1]),float(split_data[2]),float(split_data[3]), type_map[split_data[4]]]
        except:
            pass
        data.append(output_row)

data_points = len(data)

print("Data header")
data[:5]

Data header


[[5.9, 3.0, 5.1, 1.8, 2],
 [5.1, 3.5, 1.4, 0.2, 0],
 [4.9, 3.0, 1.4, 0.2, 0],
 [4.7, 3.2, 1.3, 0.2, 0],
 [4.6, 3.1, 1.5, 0.2, 0]]

In [987]:
max_value = 0
min_value = float('inf')
for i in range(0, len(data)):
    if max(data[i][:4]) > max_value:
        max_value = max(data[i][:4])
    if min(data[i][:4]) < min_value:
        min_value = min(data[i][:4])

delta = max_value - min_value
data = [[ [((data[d][v] - min_value) / delta), data[d][v]][v==4] for v in range(len(data[d]))] for d in range(len(data))]

In [988]:
random.shuffle(data)
input_data, output_data = [], []
for i in range(data_points):
    input_data.append(data[i][:4])
    output_data.append([[0, 1][t == data[i][4]] for t in range(3)])

In [989]:
train_x, train_y = input_data[:math.floor(data_points * 0.7)], output_data[:math.floor(data_points * 0.7)]
test_x, test_y = input_data[math.floor(data_points * 0.7):], output_data[math.floor(data_points * 0.7):]

## Create Network

In [1076]:
network_shape = [4,5,3]
initial_weight_range = 0.1

network_weights = create_network_weights(network_shape, initial_weight_range)

functions = [leaky_relu]
functions_p = [leaky_relu_p]

learning_rate = 0.01

In [1077]:
current_loss = 0
current_error = 0

total_backpropogations = len(train_x) * 500
backpropogations_complete = 0

print("Starting training...")
print("Network shape: " + str(network_shape))
print("Initial weight range: " + str(-1 * initial_weight_range) + " to " + str(str(initial_weight_range)))
print("Inner layer activation functions: " + "".join(str(functions)))
print("Learning rate: " + str(learning_rate))
print("\n\n")

while backpropogations_complete < total_backpropogations:

    train_index = backpropogations_complete % len(train_x)
    
    network_weights, current_loss, current_error = back_propogate_network(train_x[train_index], network_weights, functions, functions_p, train_y[train_index])
    backpropogations_complete += 1
    
    if backpropogations_complete % 500 == 0 or backpropogations_complete == total_backpropogations:
            network_test = test_network(test_x, test_y, network_weights)
            print("Backpropogations complete: " + str(backpropogations_complete) + "/" + str(total_backpropogations))
            print("     Network weights: " + str(network_weights))
            print("     Loss: " + str(current_loss))
            print("     Latest error: " + str(current_error))
            print("     Accuracy: " + str(round(network_test[0]/sum(network_test)*100,2)) + " ("+str(network_test[0])+"/"+str(sum(network_test))+")")
            print("\n\n")

Starting training...
Network shape: [4, 5, 3]
Initial weight range: -0.1 to 0.1
Inner layer activation functions: [<function leaky_relu at 0x0000021FFC8CE340>]
Learning rate: 0.01



Backpropogations complete: 500/52500
     Network weights: [[[-0.05632220614084325, -0.08023868426463672, -0.07991046045062727, 0.014252543812595284], [-0.05070744563135482, -0.07584586223859885, 0.0587864130401601, 0.041512365310085855], [0.03767887844813407, 0.0023371074436366124, 0.02016035574368628, 0.006129702769168947], [-0.06278790494306133, 0.052014182132845324, 0.015679296346564127, -0.06819049907976624], [0.08324555101734502, -0.0003844506250250888, -0.01688839770913479, 0.04928585860025436]], [[0.07978377493276485, -0.07864166079615281, 0.05077860848340392, 0.0900212443485021, 0.029342423345201492], [-0.09175922831161253, 0.09935707322348683, 0.0556076731668656, 0.06281018129152967, -0.0025426290976592095], [0.09023475555156725, 0.02725405601739544, 0.08118663541646598, 0.02350647552762438, 0.08